In [1]:
import scanpy as sc
import torch
import numpy as np
import sys
sys.path.insert(0, "../")
from cancerfoundation.model.model import CancerFoundation
import json
from cancerfoundation.data.preprocess import binning
import pandas as pd

model_name = "my_init_4gpu_lrx2"

# Load model
with open(f"../save/{model_name}/vocab.json", "r") as f:
    vocab = json.load(f)

model = CancerFoundation.load_from_checkpoint(
    f'../save/{model_name}/epoch_epoch=14.ckpt', 
    vocab=vocab, 
    strict=True
)
model.eval()
model.cuda()

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CancerFoundation(
  (criterion): MSE()
  (model): OptimizedModule(
    (_orig_mod): TransformerModule(
      (value_encoder): TheirContinuousValueEncoder(
        (dropout): Dropout(p=0.2, inplace=False)
        (linear1): Linear(in_features=1, out_features=256, bias=True)
        (activation): ReLU()
        (linear2): Linear(in_features=256, out_features=256, bias=True)
        (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      )
      (criterion_conditions): CrossEntropyLoss()
      (criterion): MSE()
      (condition_encoders): ModuleDict(
        (technology): ConditionEncoder(
          (embedding): Embedding(15, 256)
          (enc_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        )
      )
      (transformer_encoder): CFGenerator(
        (layers): ModuleList(
          (0-5): 6 x CFLayer(
            (self_attn): MHA(
              (self_attn): MultiheadAttention(
                (out_proj): NonDynamicallyQuantizableLinear(in_features=256, o

In [2]:
data = sc.read_h5ad("../proteome/PBMC_testset_preprocessed.h5ad")
data.X = data.X.toarray()

In [3]:
common_genes = list(set(vocab.keys()).intersection(set(data.var.index)))
data = data[:, common_genes].copy()

sc.pp.highly_variable_genes(
    data, 
    n_top_genes=1200, 
    layer=None  # Operate on data.X
)
data = data[:, data.var['highly_variable']]
data.X

ArrayView([[0.      , 0.      , 0.      , ..., 0.      , 0.      ,
            0.      ],
           [0.      , 0.      , 0.      , ..., 0.      , 0.      ,
            0.      ],
           [0.      , 0.      , 0.      , ..., 0.      , 0.      ,
            0.      ],
           ...,
           [0.      , 0.      , 0.      , ..., 0.      , 0.      ,
            0.      ],
           [0.      , 0.      , 0.      , ..., 0.      , 1.687557,
            0.      ],
           [0.      , 0.      , 0.      , ..., 0.      , 0.      ,
            0.      ]], dtype=float32)

In [4]:
if data.is_view:
    data = data.copy()
data.X

array([[0.      , 0.      , 0.      , ..., 0.      , 0.      , 0.      ],
       [0.      , 0.      , 0.      , ..., 0.      , 0.      , 0.      ],
       [0.      , 0.      , 0.      , ..., 0.      , 0.      , 0.      ],
       ...,
       [0.      , 0.      , 0.      , ..., 0.      , 0.      , 0.      ],
       [0.      , 0.      , 0.      , ..., 0.      , 1.687557, 0.      ],
       [0.      , 0.      , 0.      , ..., 0.      , 0.      , 0.      ]],
      dtype=float32)

In [5]:
for idx in range(data.n_obs):
    data.X[idx] = binning(data.X[idx], 50)
    if model.model.decoder.normalise_bins:
        data.X[idx] = data.X[idx] / 50

In [6]:
# Prepare tensors manually
gene_ids = torch.LongTensor([vocab[g] for g in data.var.index])
count_matrix = data.X if isinstance(data.X, np.ndarray) else data.X.toarray()

# Embed in batches
batch_size = 16

In [7]:
with torch.no_grad():
    batch_expr = torch.FloatTensor(count_matrix[0:batch_size]).cuda()
    batch_genes = gene_ids.unsqueeze(0).expand(batch_expr.shape[0], -1).cuda()

    # Add CLS token
    cls_id = vocab["<cls>"]
    batch_genes = torch.cat([
        torch.full((batch_expr.shape[0], 1), cls_id, dtype=torch.long, device='cuda'),
        batch_genes
    ], dim=1)
    batch_expr = torch.cat([
        torch.full((batch_expr.shape[0], 1), -2, device='cuda'),  # pad_value for CLS
        batch_expr
    ], dim=1)
    padding_mask = torch.zeros(batch_genes.shape, dtype=torch.bool, device='cuda')

    emb = model.model.embed(
    src=batch_genes,
    values=batch_expr,
    src_key_padding_mask=padding_mask
        )[0]

In [10]:
emb[0, 0]

tensor([ 0.6555, -0.0967, -1.3960, -0.0527, -2.0881, -0.2277, -0.5783, -0.0265,
         0.3561,  0.6471, -0.7005, -0.1086,  0.4601, -1.2243,  0.0681, -1.5785,
        -1.3406,  0.4208, -0.0743, -0.3654,  1.0748,  0.2700,  0.0237,  0.1744,
        -0.1402,  0.4368, -0.0787, -1.1794,  0.6223, -0.1522, -0.0863,  0.0059,
         0.6225, -1.4726,  1.5173, -1.5637,  0.6354,  0.4797, -1.6041,  0.9649,
         0.7411, -0.4429,  0.0686,  0.1609, -0.3209, -0.8833,  0.1037, -0.7058,
         0.2093,  0.2737,  0.2799,  0.4467,  1.3594, -0.4716,  0.9378, -0.3226,
        -0.1539,  1.0863,  0.6588,  2.9771,  0.0161,  1.7081, -0.6393, -0.0867,
        -1.5650,  0.5826,  0.8933,  0.0841,  0.2546, -1.2331,  0.0614,  0.6051,
        -1.0198, -0.8900,  1.0690,  0.0104, -0.7358,  0.8271, -0.9868,  0.9234,
         0.3393, -1.5592,  2.5755, -1.0303,  0.1992, -0.0580,  0.3886, -0.1903,
        -1.4500, -0.1628,  0.5810,  0.1688, -0.6865,  0.5229,  0.3331,  0.5651,
        -0.9375, -0.1250,  0.2475, -0.27

In [11]:
def reset_all_weights(model):
    """Recursively reset all parameters in the model"""
    @torch.no_grad()
    def init_weights(m):
        if hasattr(m, 'reset_parameters'):
            m.reset_parameters()
    
    model.apply(init_weights)

In [12]:
reset_all_weights(model)

In [13]:
with torch.no_grad():
    batch_expr = torch.FloatTensor(count_matrix[0:batch_size]).cuda()
    batch_genes = gene_ids.unsqueeze(0).expand(batch_expr.shape[0], -1).cuda()

    # Add CLS token
    cls_id = vocab["<cls>"]
    batch_genes = torch.cat([
        torch.full((batch_expr.shape[0], 1), cls_id, dtype=torch.long, device='cuda'),
        batch_genes
    ], dim=1)
    batch_expr = torch.cat([
        torch.full((batch_expr.shape[0], 1), -2, device='cuda'),  # pad_value for CLS
        batch_expr
    ], dim=1)
    padding_mask = torch.zeros(batch_genes.shape, dtype=torch.bool, device='cuda')

    emb = model.model.embed(
    src=batch_genes,
    values=batch_expr,
    src_key_padding_mask=padding_mask
        )[0]

In [14]:
emb[0, 0]

tensor([-0.7390,  0.3727,  0.7364, -1.3911,  0.4674, -0.9487,  1.7383,  0.2965,
        -0.8955,  0.5638,  1.2613,  0.6716,  1.2891, -1.9422,  0.4023, -0.1903,
        -0.7523,  0.1645,  0.7029,  0.6054,  1.8485,  1.7436,  0.1189, -1.0493,
         1.3959, -0.4928, -0.4242, -0.3079,  0.2776, -1.7238,  0.6208,  2.8256,
        -0.3873, -1.0098,  1.0445, -0.4717,  0.2132,  2.0000, -0.8241,  0.5619,
         1.2427,  0.5151,  0.4963, -0.5117,  1.8174,  1.5767, -1.7841, -0.3033,
         0.8781,  0.4685, -1.2199, -0.3188,  0.6237, -0.7959, -0.5943, -0.1057,
        -1.2040, -0.8050, -0.7251, -0.2689,  1.0165,  0.0975,  1.1718,  0.1575,
         0.4518, -1.9795, -1.3635, -1.3939, -0.1862, -0.1594,  0.8637,  0.2106,
        -1.1059,  0.3109, -0.1210,  1.3056, -0.8398, -0.2942, -0.9407, -2.0671,
        -0.1027, -0.4113, -0.8576, -0.8078, -0.1450, -0.2689,  1.4416, -0.9456,
         0.8351,  1.2788, -1.7997, -0.0642,  0.6876, -0.8261, -0.4926, -0.5916,
         0.6779, -1.3737,  1.0524,  0.42

In [7]:
batch_expr = torch.FloatTensor(count_matrix[0:batch_size]).cuda()
batch_genes = gene_ids.unsqueeze(0).expand(batch_expr.shape[0], -1).cuda()

# Add CLS token
cls_id = vocab["<cls>"]
batch_genes = torch.cat([
    torch.full((batch_expr.shape[0], 1), cls_id, dtype=torch.long, device='cuda'),
    batch_genes
], dim=1)
batch_expr = torch.cat([
    torch.full((batch_expr.shape[0], 1), -2, device='cuda'),  # pad_value for CLS
    batch_expr
], dim=1)

# Create padding mask (False = not padded)
padding_mask = torch.zeros(batch_genes.shape, dtype=torch.bool, device='cuda')
print(batch_expr.shape)
# Get embeddings from transformer
emb = model.model.embed(
    src=batch_genes,
    values=batch_expr,
    src_key_padding_mask=padding_mask
        )[0]

torch.Size([64, 1201])


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.75 GiB. GPU 0 has a total capacity of 23.54 GiB of which 2.63 GiB is free. Process 1876993 has 20.75 GiB memory in use. Of the allocated memory 17.93 GiB is allocated by PyTorch, and 2.38 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [19]:
embeddings = []
with torch.no_grad():
    
    for i in range(0, len(data), batch_size):
        batch_expr = torch.FloatTensor(count_matrix[i:i+batch_size]).cuda()
        batch_genes = gene_ids.unsqueeze(0).expand(batch_expr.shape[0], -1).cuda()
        
        # Add CLS token
        cls_id = vocab["<cls>"]
        batch_genes = torch.cat([
            torch.full((batch_expr.shape[0], 1), cls_id, dtype=torch.long, device='cuda'),
            batch_genes
        ], dim=1)
        batch_expr = torch.cat([
            torch.full((batch_expr.shape[0], 1), -2, device='cuda'),  # pad_value for CLS
            batch_expr
        ], dim=1)
        
        # Create padding mask (False = not padded)
        padding_mask = torch.zeros(batch_genes.shape, dtype=torch.bool, device='cuda')
        print(batch_expr.shape)
        # Get embeddings from transformer
        emb = model.model.embed(
            src=batch_genes,
            values=batch_expr,
            src_key_padding_mask=padding_mask
        )[0]
        
        # Extract CLS token embedding (cell embedding)
        cell_emb = emb[:, 0, :]
        embeddings.append(cell_emb.cpu().numpy())

data.obsm["CancerFoundation"] = np.concatenate(embeddings, axis=0)
print(f"Embedded {data.shape[0]} cells with dimension {data.obsm['CancerFoundation'].shape[1]}")

torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size([64, 1201])
torch.Size(

In [20]:
embeddings = pd.DataFrame(data.obsm["CancerFoundation"], index=data.obs_names, columns=[f"dim_{i}" for i in range(data.obsm["CancerFoundation"].shape[1])])

In [21]:
# data.write_h5ad(f"./proteome/{model_name}_prot_embeddings.h5ad")
embeddings.to_csv(f"./proteome/{model_name}_PBMC_testset.csv")